## `pivot_table` — reestructurar tablas

Un `groupby` con dos columnas devuelve una **Serie**, no una tabla:
crea un indice cuya etiqueta tiene dos piezas — `('Adelie', 'Biscoe')` —
y una sola columna de numeros al lado.
En pantalla parece una tabla de dos columnas, pero `shape` delata que
solo tiene una dimension.

`pivot_table` coge esos mismos numeros y los coloca en una cuadricula:
una variable hacia abajo y la otra hacia el lado. **No calcula nada distinto,
solo mueve la informacion de sitio.**

| Argumento | Que indica |
|---|---|
| `index` | que variable va en las **filas** |
| `columns` | que variable va en las **columnas** |
| `values` | la columna **en bruto** de la que salen los numeros |
| `aggfunc` | el calculo que reduce esos numeros a uno solo por casilla |

`values` dice **que** se mide, `aggfunc` dice **como** se resume.
Hace falta `aggfunc` porque en cada casilla no hay un pinguino, hay decenas,
y en la casilla solo cabe un numero.

### Los NaN de la cuadricula

Al pasar de lista a rejilla, pandas dibuja **todas** las combinaciones posibles
(3 especies x 3 islas = 9 casillas), existan o no. La Serie de origen tenia
5 filas y ninguna vacia: era la lista de lo que existe.

Una casilla vacia aqui significa **que no hay pinguinos de esa especie en esa isla**.
No es una medicion que falte: es una combinacion que no existe.

Por eso rellenarla con 0 seria mentira — no es "pesan 0 gramos",
es "no los hay". Seria inventar pinguinos.

**Tercer tipo de NaN del catalogo:**

| Tipo | Origen | Que hacer |
|---|---|---|
| NaN honesto | el dato no se midio | valorar si imputar |
| NaN por fallo de estructura | el dato existe pero fue a otra columna | arreglar la estructura |
| NaN estructural de rejilla | la combinacion no existe | dejarlo, no rellenar |


In [1]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset('penguins')
df.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


In [2]:
resultado = df.groupby(['species', 'island'])['body_mass_g'].mean()
print(type(resultado))
print(resultado.shape)
resultado.index


<class 'pandas.core.series.Series'>
(5,)


MultiIndex([(   'Adelie',    'Biscoe'),
            (   'Adelie',     'Dream'),
            (   'Adelie', 'Torgersen'),
            ('Chinstrap',     'Dream'),
            (   'Gentoo',    'Biscoe')],
           names=['species', 'island'])

In [3]:
tabla = df.pivot_table(
    index='species',
    columns='island',
    values='body_mass_g',
    aggfunc='mean'
)
tabla

island,Biscoe,Dream,Torgersen
species,,,
Adelie,3709.659091,3688.392857,3706.372549
Chinstrap,NaN,3733.088235,NaN
Gentoo,5076.016260,NaN,NaN


### Error silencioso nº 25: `fill_value` mal aplicado en `pivot_table`

Cuando una casilla de la rejilla no tiene ninguna fila detras,
pandas **no calcula nada** y deja `NaN`. La casilla vacia es un hueco
de la forma de la tabla, no un resultado.

Con `count` y `sum` eso esta **mal**: contar cero filas da `0`,
sumar cero filas da `0`. Y la diferencia no es cosmetica:

```python
conteo['Dream'].mean()      # (56 + 68) / 2  = 62      <- divide entre 2
conteo_ok['Dream'].mean()   # (56 + 68 + 0) / 3 = 41.3 <- divide entre 3
```

`.mean()` **ignora los NaN**. La especie que faltaba se cae de la cuenta
sin ningun aviso y sale un numero creible pero falso.
Se arregla con `fill_value=0`.

Con `mean`, `median`, `std`, `min` o `max` es al reves: un `0` ahi
seria un dato **inventado** ("los Gentoo de Dream pesan 0 gramos").
Ademas de mentir, arrastra las medias hacia abajo y gana cualquier
busqueda de minimos. Aqui el `NaN` es la respuesta honesta.

| `aggfunc` | Casilla sin filas | Por que |
|---|---|---|
| `count`, `sum` | `fill_value=0` | contar o sumar nada da cero |
| `mean`, `median`, `std`, `min`, `max` | dejar `NaN` | de nada no sale un promedio |

**La pregunta a hacerse:** ¿el cero es un valor verdadero para este calculo,
o me lo estoy inventando?

**Y el aviso de fondo:** que tu distingas el `NaN` del `0` mirando la tabla
no sirve de nada. El codigo que venga despues no lo distingue.

In [4]:
conteo = df.pivot_table(
    index='species',
    columns='island',
    values='body_mass_g',
    aggfunc='count'
)
conteo

island,Biscoe,Dream,Torgersen
species,,,
Adelie,44.0,56.0,51.0
Chinstrap,NaN,68.0,NaN
Gentoo,123.0,NaN,NaN


In [5]:
suma = df.pivot_table(
    index='species',
    columns='island',
    values='body_mass_g',
    aggfunc='sum'
)
suma

island,Biscoe,Dream,Torgersen
species,,,
Adelie,163225.0,206550.0,189025.0
Chinstrap,NaN,253850.0,NaN
Gentoo,624350.0,NaN,NaN


In [6]:
conteo_ok = df.pivot_table(
    index='species',
    columns='island',
    values='body_mass_g',
    aggfunc='count',
    fill_value=0
)
conteo_ok

island,Biscoe,Dream,Torgersen
species,,,
Adelie,44,56,51
Chinstrap,0,68,0
Gentoo,123,0,0


### Error silencioso nº 26: filas fantasma tras `pivot_table` + `melt`

La ida y la vuelta **no devuelven al punto de partida**:

Serie original -> 5 filas (lo que existe)  
pivot_table -> cuadricula 3x3 = 9 casillas, 4 vacias  
melt -> 9 filas <- las 4 vacias se han vuelto filas reales  


Los huecos que invento la rejilla se quedan a vivir como filas.
Son combinaciones que **no existen en los datos**, pero ocupan sitio.

**Por que es grave:** `size()` cuenta filas, no mira lo que hay dentro.
Una fila con `NaN` cuenta igual que una fila con dato.

```python
largo.groupby('species').size()      # Adelie 3, Chinstrap 3, Gentoo 3
```

Todas a 3. La verdad es Adelie 3, Chinstrap 1, Gentoo 1.
Y "cada especie vive en 3 islas" es una frase **creible**: no chirria,
no levanta sospechas, se publica.

**Solucion:** `largo.dropna()` despues del melt. De 9 vuelve a 5.

**De paso, dos metodos que se parecen y no son lo mismo:**

| Metodo | Que cuenta |
|---|---|
| `.size()` | filas, incluidas las que son todo `NaN` |
| `.count()` | solo valores no nulos |

In [7]:
tabla_plana = tabla.reset_index()
tabla_plana

island,species,Biscoe,Dream,Torgersen
0,Adelie,3709.659091,3688.392857,3706.372549
1,Chinstrap,NaN,3733.088235,NaN
2,Gentoo,5076.016260,NaN,NaN


In [8]:
largo = tabla_plana.melt(
    id_vars='species',
    var_name='island',
    value_name='peso_medio'
)
largo

,species,island,peso_medio
0,Adelie,Biscoe,3709.659091
1,Chinstrap,Biscoe,NaN
2,Gentoo,Biscoe,5076.016260
3,Adelie,Dream,3688.392857
4,Chinstrap,Dream,3733.088235
5,Gentoo,Dream,NaN
6,Adelie,Torgersen,3706.372549
7,Chinstrap,Torgersen,NaN
8,Gentoo,Torgersen,NaN


In [9]:
largo_limpio = largo.dropna()
print(largo.shape)
print(largo_limpio.shape)

(9, 3)
(5, 3)


### Ejercicio: de la cuadricula al formato largo

Objetivo: contar pinguinos por especie e isla, pasando por la rejilla
y volviendo a formato largo.

#### 1. `size` y no `count`

| Verbo | Que mira | Necesita `values` |
|---|---|---|
| `size` | la fila entera | no |
| `count` | dentro de una columna | si |

Contar pinguinos es contar **filas**, y una fila es un pinguino aunque
le falten mediciones. Por eso `size`.

El problema de `count` no es solo que descarte nulos: es que **obliga a
elegir una columna**, y el numero depende de cual elijas.
`count` sobre `body_mass_g` da un resultado, sobre `sex` da otro.
Misma pregunta, dos respuestas.

> **Senal de alarma:** si el resultado cambia segun una columna que a mi
> no me interesa, no estoy midiendo lo que creo que mido.

#### 2. Cuando el 0 es verdad

Misma rejilla, mismos huecos, decision contraria segun el `aggfunc`:

| Sesion | `aggfunc` | Casilla vacia | Por que |
|---|---|---|---|
| 6 | `mean` | `NaN` | de cero mediciones no sale ningun promedio |
| 7 | `size` | `0` | contar cero filas da cero |

**La pregunta general:** ¿el cero **sale del calculo**, o lo **pongo yo**
para tapar un hueco?

Contar nada da cero. Sumar nada da cero. Promediar nada no da nada.

#### 3. Las filas fantasma no siempre son `NaN`

Al tapar los huecos con `fill_value=0`, las 4 combinaciones inexistentes
llegan al formato largo **con un 0 dentro**, no con un `NaN`.

Un `0` es un valor legitimo: `dropna()` no lo toca. Hay que filtrarlas
a mano con una mascara (`!= 0`).

**Son mas dificiles de detectar que las de la sesion 6, no menos:**
ya no llevan ninguna marca que las delate.

Y si sobran o no **depende de la pregunta**:

| Pregunta | Las 9 filas |
|---|---|
| ¿Cuantos pinguinos hay de cada combinacion? | correctas — `Chinstrap-Biscoe = 0` es verdad |
| ¿En que islas vive cada especie? | mienten — salen 3 islas para todas |

La misma tabla contesta bien a una y mal a la otra.

#### 4. Dos comprobaciones, no una

| Comprobacion | Que detecta |
|---|---|
| `.shape` | filas que se caen o se duplican |
| suma de la columna vs `len(df)` | datos perdidos por el camino |

**La trampa:** al filtrar se fueron 4 filas y la suma **no se movio**,
porque valian 0.

> Una suma que cuadra **no** garantiza que el numero de filas sea correcto.

Cada comprobacion pilla lo que la otra no ve. Siempre las dos.

**Regla de fondo:** reestructurar mueve los datos de sitio, no los crea
ni los destruye. Si algo cambia, se ha perdido o duplicado — y casi nunca
va a saltar un error.

In [10]:
tabla_2 = df.pivot_table(
    index='species',
    columns='island',
    aggfunc='size',
    fill_value=0
)
tabla_2


island,Biscoe,Dream,Torgersen
species,,,
Adelie,44,56,52
Chinstrap,0,68,0
Gentoo,124,0,0


In [11]:
print(tabla_2.values.sum())
len(df)

344


344

In [12]:
largo_2 = tabla_2.reset_index().melt(
    id_vars='species',
    var_name='island',
    value_name='numero_pinguinos'
)
largo_2

,species,island,numero_pinguinos
0,Adelie,Biscoe,44
1,Chinstrap,Biscoe,0
2,Gentoo,Biscoe,124
3,Adelie,Dream,56
4,Chinstrap,Dream,68
5,Gentoo,Dream,0
6,Adelie,Torgersen,52
7,Chinstrap,Torgersen,0
8,Gentoo,Torgersen,0


In [13]:
largo2_filtrado = largo_2[largo_2["numero_pinguinos"] != 0] 
largo2_filtrado

,species,island,numero_pinguinos
0,Adelie,Biscoe,44
2,Gentoo,Biscoe,124
3,Adelie,Dream,56
4,Chinstrap,Dream,68
6,Adelie,Torgersen,52


In [14]:
largo2_filtrado.shape

(5, 3)